## Copying repo from git (FOR GOOGLE COLAB ONLY!!!)

In [9]:
import os
import sys

!git clone https://github.com/isachagit/ML-DS-pet-projects/JerryDetect

%cd JerryDetect

sys.path.append(os.getcwd())

!nvidia-smi


fatal: destination path 'JerryDetect' already exists and is not an empty directory.
/content/ML-DS-pet-projects/JerryDetect
Thu Sep  3 17:24:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|               

## Lib Installation & Import

In [10]:
import os
import subprocess
import sys


def install_requirements(file_path="requirements.txt"):
    if not os.path.exists(file_path):
        print("Missing", file_path)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", file_path])
        print("Successfully installed")
    except subprocess.CalledProcessError as e:
        print(f"Error: {e}")

install_requirements()

In [11]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from PIL import Image, ImageDraw, ImageFont
import torchvision.transforms.v2 as transforms
import cv2
import numpy as np
from torch.utils.data import Dataset

## CNN Model init

In [13]:
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

COCO_INSTANCE_CATEGORY_NAMES = weights.meta["categories"]
COCO_INSTANCE_CATEGORY_NAMES[18] = 'Jerry' #Separate class for my dog
cnn_model = fasterrcnn_resnet50_fpn(weights=weights)

## Custom dataset

In [16]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, targets, transforms=None):
        self.image_paths = image_paths
        self.targets = targets
        self.transforms = transforms

    def __getitem__(self, idx):
        img = Image.open(r"pics_train/"+self.image_paths[idx]).convert("RGB")
        img = transforms.functional.to_image(img)
        img = transforms.functional.to_dtype(img, scale=True)
        target = self.targets[idx]

        if self.transforms is not None:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.image_paths)

image_paths = ["photo_2026-06-01_18-16-30.jpg", "photo_2026-06-01_18-16-19.jpg", "photo_2026-06-01_18-15-47.jpg", "barbos1.jpg", "barbos2.jpg", "barbos3.jpg", "barbos4.jpg", "barbos5.jpg", "barbos6.jpg", "barbos7.jpg", "barbos8.jpg"]

tgt = [
    { 'boxes' : torch.tensor([[485, 97, 606, 186]]),
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[274, 184, 355, 339]]),
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[395, 485, 592, 772]]),
      'labels' : torch.tensor([18])
    }
    ,
    { 'boxes' : torch.tensor([[884, 356, 1185, 644]]),
      'labels' : torch.tensor([18])
    }
      ,
    { 'boxes' : torch.tensor([[935, 384, 1085, 682]]),
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[748, 410, 1156, 636]]),
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[728, 411, 1137, 683]]),
      'labels' : torch.tensor([18])
    }
    ,
    { 'boxes' : torch.tensor([[686, 424, 1086, 811]]),
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[901, 433, 1093, 684]]),
      'labels' : torch.tensor([18])
    }
    ,
    { 'boxes' : torch.tensor([[940, 443, 1113, 674]]),
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[828, 306, 1136, 560]]),
      'labels' : torch.tensor([18])
    }
]

dog_fine = CustomDataset(image_paths, tgt)



## Training

In [17]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('Device: ', device)
cnn_model.to(device)

params = [p for p in cnn_model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=0.005)

def collate_fn(batch):
    return tuple(zip(*batch))

cnn_model.train()

data_loader = torch.utils.data.DataLoader(
    dog_fine, batch_size=2, shuffle=True, collate_fn=collate_fn
)

for epoch in range(200):
    for images, targets in data_loader:
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = cnn_model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    print(f"Epoch {epoch} loss: {losses.item()}")


Device:  cuda
Epoch 0 loss: 0.020870540291070938
Epoch 1 loss: 0.02807425521314144
Epoch 2 loss: 0.04886830970644951
Epoch 3 loss: 0.024452921003103256
Epoch 4 loss: 0.03283337131142616
Epoch 5 loss: 0.020881501957774162
Epoch 6 loss: 0.020478561520576477
Epoch 7 loss: 0.015835532918572426
Epoch 8 loss: 0.018304625526070595
Epoch 9 loss: 0.018004121258854866
Epoch 10 loss: 0.03248342499136925
Epoch 11 loss: 0.01810253970324993
Epoch 12 loss: 0.018588630482554436
Epoch 13 loss: 0.016574377194046974
Epoch 14 loss: 0.02056725323200226
Epoch 15 loss: 0.02237718179821968
Epoch 16 loss: 0.04729257524013519
Epoch 17 loss: 0.01069572288542986
Epoch 18 loss: 0.014688819646835327
Epoch 19 loss: 0.022149216383695602
Epoch 20 loss: 0.02378319762647152
Epoch 21 loss: 0.01867019757628441
Epoch 22 loss: 0.01045943796634674
Epoch 23 loss: 0.007308728527277708
Epoch 24 loss: 0.006182218436151743
Epoch 25 loss: 0.01532678585499525
Epoch 26 loss: 0.012226252816617489
Epoch 27 loss: 0.009901897981762886
E

## On-video detection

In [35]:
cnn_model.rpn.nms_thresh = 0.1
cnn_model.roi_heads.nms_thresh = 0.01
cnn_model.eval()

vids_detected_path = r"vids_detected/"
vids_detected_path += "jerry_cv.mp4"
vids_test_path = r"vids_test/" + r"jerry_2_2.mp4"

def video_to_frames(video_path, frame_rate):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    saved_count = 0
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    video = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_rate == 0:
            rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(rgb_image)
            img_tensor = F.to_tensor(pil_image).unsqueeze(0).to(device)

            with torch.no_grad():
                predictions = cnn_model(img_tensor)

            boxes = predictions[0]['boxes']
            labels = predictions[0]['labels']
            scores = predictions[0]['scores']

            draw = ImageDraw.Draw(pil_image)
            threshold = .2
            for box, label, score in zip(boxes, labels, scores):
                class_name = COCO_INSTANCE_CATEGORY_NAMES[label]
                if score > threshold and class_name == 'Jerry':
                    xmin, ymin, xmax, ymax = box.tolist()
                    draw.rectangle([(xmin, ymin), (xmax, ymax)], outline="red", width=3)
                    draw.text((xmin, ymin - 30), f"{'Jerry'}: {score*100:.0f} %", fill="red")

            output_frame = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
            height, width, _ = output_frame.shape

            if video is None:
                video = cv2.VideoWriter(vids_detected_path, fourcc, 30, (width, height), isColor=True)

            video.write(output_frame)

            saved_count += 1
            #print('frame', saved_count * frame_rate)

        frame_count += 1

    cap.release()

    if video is not None:
        video.release()

    print(f"Frames: {saved_count} out of {frame_count}")


video_to_frames(vids_test_path, frame_rate=1)

frame 1
frame 2
frame 3
frame 4
frame 5
frame 6
frame 7
frame 8
frame 9
frame 10
frame 11
frame 12
frame 13
frame 14
frame 15
frame 16
frame 17
frame 18
frame 19
frame 20
frame 21
frame 22
frame 23
frame 24
frame 25
frame 26
frame 27
frame 28
frame 29
frame 30
frame 31
frame 32
frame 33
frame 34
frame 35
frame 36
frame 37
frame 38
frame 39
frame 40
frame 41
frame 42
frame 43
frame 44
frame 45
frame 46
frame 47
frame 48
frame 49
frame 50
frame 51
frame 52
frame 53
frame 54
frame 55
frame 56
frame 57
frame 58
frame 59
frame 60
frame 61
frame 62
frame 63
frame 64
frame 65
frame 66
frame 67
frame 68
frame 69
frame 70
frame 71
frame 72
frame 73
frame 74
frame 75
frame 76
frame 77
frame 78
frame 79
frame 80
frame 81
frame 82
frame 83
frame 84
frame 85
frame 86
frame 87
frame 88
frame 89
frame 90
frame 91
frame 92
frame 93
frame 94
frame 95
frame 96
frame 97
frame 98
frame 99
frame 100
frame 101
frame 102
frame 103
frame 104
frame 105
frame 106
frame 107
frame 108
frame 109
frame 110
frame 11